# 📚 Resumo: RAG - Retrieval Augmented Generation

---

## 🔎 O que é RAG?

| Termo | Significado |
|-------|-------------|
| Retrieval | Recuperação de informações relevantes |
| Augmented | Incorpora informações externas |
| Generation | Geração de texto |

> Em resumo: O RAG busca informações relevantes e as usa para enriquecer a resposta do modelo de IA. 🎯

---

## ⚙️ Pipeline no Notebook

PDF →Chunks →Vetores (Embeddings) →Indexação (FAISS) →Retriever →Prompt →LLM


1. Carregar PDF e quebrar em chunks (pedaços)
2. Transformar em vetores com modelo de embedding
3. Indexar com FAISS (busca por similaridade)
4. Criar retriever para buscar documentos relevantes

---

## 🤖 Embeddings

### Modelos de Embedding

| Modelo | Características |
|--------|-----------------|
| nomic-embed-text | Leve, do Ollama |
| mxbai-embed-large | Maior precisão |
| paraphrase-multilingual-MiniLM-L12-v2 | Multilíngue (HuggingFace) |

### Por que HuggingFace?

- Modelos *:cloud (como minimax-m2.5:cloud) são apenas para chat, não para embeddings
- Por isso o notebook usa HuggingFace para gerar vetores 📊

---

## 🔧 Comandos Importantes

# Normalizar vetores (similaridade de cosseno)
encode_kwargs={"normalize_embeddings": True}

# Quantos documentos buscar
search_kwargs={"k": 4}


---

## 💡 Em Uma Frase

> Esse bloco é a parte "Retrieval" do RAG — converte pedaços do PDF em vetores, indexa com FAISS e prepara a busca que alimenta o contexto da resposta do LLM. 🚀

In [25]:
from os import path

from langchain_community.chat_models import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [26]:
path_file = path.join('/', 'tmp', 'Explorando o Universo das IAs com Hugging Face.pdf')
pdf = PyPDFLoader(path_file)
docs = pdf.load()

len(docs)

89

In [27]:
llm = ChatOllama(model="minimax-m2.5:cloud", temperature=0.1)

In [28]:
docs[0].page_content

'Explorando o Universo das IAs com\nHugging Face\nAsimov Academy'

In [29]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [30]:
# nomic-embed-text precisa de `ollama pull`; modelos *:cloud* não servem para embeddings.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1133.10it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
prompt = ChatPromptTemplate.from_template(
    """Responda baseado apenas no contexto abaixo.

Contexto: {context}

Pergunta: {question}

Resposta:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# 5. Pergunta
response = rag_chain.invoke("Qual o tema do documento?")
print(response.content)

O tema do documento é **como usar e implementar modelos de linguagem do Hugging Face**, com foco em:

- Acesso e configuração de modelos
- Classificação de texto (finanças, ironia em tweets, zero-shot)
- Implementação via código (pipelines)
- Deploy de modelos
- Formatação de tokens para instruções

Em resumo: um guia/tutorial sobre utilização de modelos de IA para classificação de texto.


In [32]:
response = rag_chain.invoke("Quém é o autor?")    
print(response.content)

Com base apenas no contexto fornecido, não há uma menção explícita de um autor específico para o conteúdo.

No entanto, a entidade mais consistente que aparece no contexto é **Asimov Academy**, que é mencionada:

1. No início: "Explorando o Universo das IAs com Hugging Face **Asimov Academy**"
2. No final: "Asimov Academy 61"

Portanto, considerando apenas as informações presentes no contexto, **Asimov Academy** parece ser a fonte/autora do material.


In [33]:
response = rag_chain.invoke("Quais as tecnologias abordadas?")
print(response.content)

Com base no contexto fornecido, as tecnologias abordadas são:

1. **ChatGPT** - Modelo de IA conversacional da OpenAI
2. **Hugging Face** - Plataforma para modelos de IA
3. **Transformers** - Biblioteca do Hugging Face para modelos de linguagem
4. **Tokenizadores** - Ferramentas para pré-processamento de texto
5. **Modelos de IA** - Modelos de linguagem e redes neurais em geral
6. **Deploy de WebApps** - Implementação de aplicações web com IA
7. **Pré-processamento e Pós-processamento** - Técnicas de tratamento de dados para IA


In [ ]:
response = rag_chain.invoke("Existe alguma forma de traduzir textos, segundo o pdf? Como seria se sim ?")
print(response.content)